# SpecLens Evaluation

Each category follows the same three steps: read files, format data, and plot/save figures. User-study task metrics and SUS share the same participants and are therefore kept together.

## Common setup

In [ ]:
%matplotlib inline

from pathlib import Path
import sys

import pandas as pd

BASE = Path.cwd()
sys.path.insert(0, str(BASE))

from utils import (
    prepare_efficiency,
    prepare_scalability,
    prepare_userstudy,
    plot_all_accuracy,
    plot_all_scalability,
    plot_all_sus,
    plot_all_time,
    plot_efficiency_benchmarks,
)

if not (BASE / "1_userstudy").is_dir():
    raise RuntimeError("Run this notebook from the datas directory.")

USERSTUDY_DIR = BASE / "1_userstudy"
EFFICIENCY_DIR = BASE / "2_efficiency"
SCALABILITY_DIR = BASE / "3_scalability"
FIGS_DIR = BASE / "figs"
USERSTUDY_FIGS = FIGS_DIR / "1_userstudy"
EFFICIENCY_FIGS = FIGS_DIR / "2_efficiency"
SCALABILITY_FIGS = FIGS_DIR / "3_scalability"
for path in (USERSTUDY_FIGS, EFFICIENCY_FIGS, SCALABILITY_FIGS):
    path.mkdir(parents=True, exist_ok=True)

## 1. User Study and SUS

### Step 1: Read files

In [ ]:
userstudy_frames = {
    "academia": pd.read_csv(USERSTUDY_DIR / "user-study-results_academia.csv"),
    "industry": pd.read_csv(USERSTUDY_DIR / "user-study-results_industry.csv"),
}

pd.DataFrame({
    "cohort": list(userstudy_frames),
    "rows": [len(frame) for frame in userstudy_frames.values()],
})

### Step 2: Format data

In [ ]:
data_by_cohort = prepare_userstudy(userstudy_frames)

pd.DataFrame([
    {"cohort": name, "participants": data["total_count"]}
    for name, data in data_by_cohort.items()
])

### Step 3: Plot and save figures

In [ ]:
userstudy_figures = [
    *plot_all_accuracy(data_by_cohort, USERSTUDY_FIGS, preview=True),
    *plot_all_time(data_by_cohort, USERSTUDY_FIGS, preview=True),
    *plot_all_sus(data_by_cohort, USERSTUDY_FIGS, preview=True),
]
print(f"Saved {len(userstudy_figures)} user-study figures to {USERSTUDY_FIGS.relative_to(BASE)}")

## 2. Efficiency

### Step 1: Read files

In [ ]:
efficiency_df = pd.read_csv(EFFICIENCY_DIR / "benchmark_summary.csv")
efficiency_df[["benchmark", "case"]].groupby("benchmark").size().rename("cases").to_frame()

### Step 2: Format data

In [ ]:
eff_x_labels, eff_series, eff_case_heights = prepare_efficiency(efficiency_df)

pd.DataFrame({
    "benchmark": eff_x_labels,
    "cases": [len(values) for values in eff_case_heights["fullsym"]],
})

### Step 3: Plot and save figures

In [ ]:
efficiency_figures = plot_efficiency_benchmarks(
    eff_x_labels,
    eff_series,
    eff_case_heights,
    EFFICIENCY_FIGS,
    preview=True,
)
print(f"Saved {len(efficiency_figures)} efficiency figure to {EFFICIENCY_FIGS.relative_to(BASE)}")

## 3. Scalability

### Step 1: Read files

In [ ]:
scalability_df = pd.read_csv(SCALABILITY_DIR / "benchmark_summary.csv")
scalability_df[["benchmark", "case"]].groupby("benchmark").size().rename("points").to_frame()

### Step 2: Format data

In [ ]:
scal_line_payloads = prepare_scalability(scalability_df)

pd.DataFrame([
    {"dataset": name, "x_values": payload["x_values"], "points": len(payload["x_values"])}
    for name, payload in scal_line_payloads.items()
])

### Step 3: Plot and save figures

In [ ]:
scalability_figures = plot_all_scalability(
    scal_line_payloads,
    SCALABILITY_FIGS,
    preview=True,
)
assert len(scalability_figures) == 3
print(f"Saved {len(scalability_figures)} scalability figures to {SCALABILITY_FIGS.relative_to(BASE)}")